# Demo 3: where do the tokens go?

Every call to the model costs **tokens** (roughly ¾ of a word each), and Groq's free
tier allows **200,000 a day**. This notebook shows, call by call, what a solve spends:

| column | what it is |
|---|---|
| **sent** | everything in the request: instructions, tool descriptions, and the whole conversation so far, so it grows every round |
| **reasoning** | the model's hidden thinking before it answers; usually the largest part |
| **answer** | what it actually says: a tool request or the worksheet |

The solver escalates through tiers to avoid spending tokens it doesn't need:
**tier 0** code only (0 tokens) → **tier 1** one model call → **tier 2** the agent
with tools.

**Setup (once)**: `uv sync --group demo`, `uv run cryptic-agent ingest`, and your
Groq key in `.env`. Then run the cells top to bottom.

In [ ]:
import json
import logging
import os
import textwrap
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from cryptic_agent.agent.ledger import ledger, request_overhead
from cryptic_agent.agent.solver import SUBMIT_TOOL, Solver, SolveResult, Step
from cryptic_agent.agent.tools import Toolbox
from cryptic_agent.lexicon.store import Lexicon
from cryptic_agent.llm.client import GroqClient
from cryptic_agent.tools.dictionary import Dictionary


def find_repo_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("run this from inside the cryptic-agent repo")


ROOT = find_repo_root(Path.cwd())
load_dotenv(ROOT / ".env")
logging.basicConfig(level=logging.ERROR)
os.environ.setdefault("CRYPTIC_AGENT_DATA_DIR", str(ROOT / "data"))

DAILY_BUDGET = 200_000  # Groq free tier, tokens per day
toolbox = Toolbox(Dictionary.load(), Lexicon())
llm = GroqClient()
solved: dict[str, SolveResult] = {}  # clue -> result, for the comparison at the end
print("ready:", llm.model)

## The helper: solve a clue and show what each step cost
`show(clue, enumeration)` prints the steps as they happen, then the ledger.

In [ ]:
def print_step(step: Step) -> None:
    if step.kind == "tier":
        print(f"\n── {step.text} ──")
    elif step.kind == "fastpass":
        lines = step.text.splitlines()
        print("fast pass (code, 0 tokens):", *lines[1:4], sep="\n   ")
    elif step.kind == "call" and step.usage:
        u = step.usage
        answer = u.completion_tokens - u.reasoning_tokens
        print(
            f"💰 {step.text}: sent {u.prompt_tokens:,} + reasoning {u.reasoning_tokens:,} "
            f"+ answer {answer:,} = {u.total_tokens:,} tokens"
        )
    elif step.kind == "thought":
        print("   💭", textwrap.shorten(step.text, 160))
    elif step.kind == "tool":
        args = ", ".join(f"{k}={v!r}" for k, v in step.arguments.items())
        print(
            f"   🔧 {step.tool}({textwrap.shorten(args, 70)}) → {textwrap.shorten(step.result, 90)}"
        )


def show(clue: str, enumeration: str, pattern: str | None = None) -> SolveResult:
    print(f"Clue: {clue} ({enumeration})")
    result = Solver(llm, toolbox, on_step=print_step).solve(clue, enumeration, pattern=pattern)
    solved[clue] = result

    verdict = result.verdict
    status = verdict.status.upper() if verdict else "NO ANSWER"
    print(f"\nANSWER: {result.answer}  [{status}]  answered at tier {result.tier}")
    if verdict:
        for check in verdict.failed():
            print(f"   ✗ {check.name}: {check.detail}")

    rows = ledger(result)
    if not rows:
        print("\nNo model calls at all: solved by code. Cost: 0 tokens.")
        return result

    table = pd.DataFrame([vars(r) for r in rows]).rename(columns={"running_total": "running total"})
    display(table)

    usage = result.usage
    total = usage.total_tokens
    answer = usage.completion_tokens - usage.reasoning_tokens
    print(f"total {total:,} tokens:")
    for name, n in [
        ("sent", usage.prompt_tokens),
        ("reasoning", usage.reasoning_tokens),
        ("answer", answer),
    ]:
        print(f"   {name:<10}{n:>7,}  {n / total:>4.0%}  {'█' * round(40 * n / total)}")
    print(
        f"= {total / DAILY_BUDGET:.1%} of a day's {DAILY_BUDGET:,} tokens; "
        f"about {DAILY_BUDGET // total} solves like this per day"
    )
    return result

## 1. An easy clue: solved by code, 0 tokens
The fast pass finds a definition ("crime") and separate wordplay (an anagram of
"Senator", signalled by "arranged") that agree. Code writes the whole parse and
checks it. The model is never called.

In [ ]:
show("Senator arranged crime", "7");

## 2. A middling clue: the fast pass finds the anagram, the model does the rest
Code finds MERINGUES as an anagram of "using mere", but nothing in the lexicon says
"sweet desserts" means MERINGUES, so it can't finish alone. One model call (tier 1)
gets the clue plus that evidence.

In [ ]:
show("Cook using mere sweet desserts", "9");

## 3. A hard clue: all the way to the agent
A full Guardian cryptic. The answer is OUNCE: a "cat" (the snow leopard), made from
B-OUNCE ("bound") with its head scratched off. Watch how the cost builds up: each
round re-sends the conversation, and the model reasons afresh every time.

In [ ]:
show("Cat bound to scratch head", "5");

## 4. The fixed cost of every request
Before any conversation, each agent request carries the instructions and a
description of every tool. These are sent again in **every** round. Sizes are
estimated at about 4 characters per token.

In [ ]:
overhead = request_overhead(toolbox)
print(
    " + ".join(f"{n:,} tokens of {what}" for what, n in overhead.items()),
    "in every tier-2 request\n",
)

tool_sizes = pd.DataFrame(
    [
        {"tool": spec["function"]["name"], "tokens (approx)": len(json.dumps(spec)) // 4}
        for spec in [*toolbox.specs(), SUBMIT_TOOL]
    ]
).sort_values("tokens (approx)", ascending=False)
display(tool_sizes)

## 5. Try your own clue
Change the clue and enumeration. Add `pattern="E?E?E?T"` for crossing letters.

In [ ]:
show("Mountain observed in severe storm", "7");

## 6. Side by side
Everything solved in this session.

In [ ]:
summary = pd.DataFrame(
    [
        {
            "clue": clue,
            "answer": r.answer,
            "verdict": r.verdict.status if r.verdict else "none",
            "tier": r.tier,
            "model calls": len(ledger(r)),
            "sent": r.usage.prompt_tokens,
            "reasoning": r.usage.reasoning_tokens,
            "answer tokens": r.usage.completion_tokens - r.usage.reasoning_tokens,
            "total": r.usage.total_tokens,
        }
        for clue, r in solved.items()
    ]
)
display(summary)
session = summary.total.sum()
print(f"this session: {session:,} tokens = {session / DAILY_BUDGET:.1%} of a day")